# 排课算法分析与调参实验

本 Notebook 用于：
- 可视化排课结果
- 分析算法性能
- 调整约束权重
- 比较不同配置的效果

## 1. 环境准备

In [ ]:
# 设置路径
import sys
import os

# 添加项目路径
project_root = os.path.dirname(os.getcwd())
backend_path = os.path.join(project_root, 'backend')
sys.path.insert(0, backend_path)

print(f"项目根目录: {project_root}")
print(f"后端路径: {backend_path}")

In [ ]:
# 导入必要模块
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 导入引擎模块
from app.engine.data.mock import generate_full_test_data, generate_simple_test_data, MockConfig, MockDataGenerator
from app.engine.state import ScheduleState
from app.engine.utils.slot_finder import SlotFinder
from app.engine.schedulers import LayerScheduler, VenueScheduler, NormalScheduler
from app.engine.evaluation import ScheduleScorer, ScheduleReporter

print("模块导入完成！")

## 2. 生成测试数据

In [ ]:
# 生成测试数据
data = generate_full_test_data()

print("数据统计:")
for key, value in data.stats.items():
    print(f"  {key}: {value}")

In [ ]:
# 查看任务分布
task_types = {
    '分层课': len([t for t in data.tasks if t.layer_group_id]),
    '场地课': len([t for t in data.tasks if t.required_venue_type and not t.layer_group_id]),
    '普通课': len([t for t in data.tasks if not t.layer_group_id and not t.required_venue_type])
}

print("任务类型分布:")
for task_type, count in task_types.items():
    print(f"  {task_type}: {count}")

# 可视化
plt.figure(figsize=(8, 6))
plt.pie(task_types.values(), labels=task_types.keys(), autopct='%1.1f%%')
plt.title('任务类型分布')
plt.show()

## 3. 运行排课算法

In [ ]:
def run_scheduling(data, verbose=True):
    """
    运行完整的排课流程
    
    Returns:
        state: 课表状态
        metrics: 运行指标
    """
    # 初始化
    state = ScheduleState()
    state.stats["total_tasks"] = len(data.tasks)
    
    # 设置场地容量
    for venue in data.venues:
        for subject in venue.subjects:
            current = state.venue_capacities.get(subject, 0)
            state.set_venue_capacity(subject, current + venue.capacity)
    
    slot_finder = SlotFinder(state, data)
    
    metrics = {}
    
    # 分层课
    if verbose:
        print("运行分层课调度器...")
    start = time.time()
    layer_scheduler = LayerScheduler(state, slot_finder, data)
    layer_ids = layer_scheduler.schedule()
    metrics['layer_time'] = time.time() - start
    metrics['layer_tasks'] = len(layer_ids)
    
    # 场地课
    if verbose:
        print("运行场地课调度器...")
    start = time.time()
    venue_scheduler = VenueScheduler(state, slot_finder, data)
    venue_ids = venue_scheduler.schedule()
    metrics['venue_time'] = time.time() - start
    metrics['venue_tasks'] = len(venue_ids)
    
    # 普通课
    if verbose:
        print("运行普通课调度器...")
    start = time.time()
    normal_scheduler = NormalScheduler(state, slot_finder, data)
    normal_ids = normal_scheduler.schedule()
    metrics['normal_time'] = time.time() - start
    metrics['normal_tasks'] = len(normal_ids)
    
    metrics['total_time'] = metrics['layer_time'] + metrics['venue_time'] + metrics['normal_time']
    metrics['total_records'] = len(state.schedule_records)
    
    return state, metrics

In [ ]:
# 运行排课
state, metrics = run_scheduling(data)

print("\n运行指标:")
print(f"  总耗时: {metrics['total_time']:.2f}s")
print(f"  分层课: {metrics['layer_tasks']} 任务, {metrics['layer_time']:.2f}s")
print(f"  场地课: {metrics['venue_tasks']} 任务, {metrics['venue_time']:.2f}s")
print(f"  普通课: {metrics['normal_tasks']} 任务, {metrics['normal_time']:.2f}s")
print(f"  总排课: {metrics['total_records']} 节")

## 4. 评分与分析

In [ ]:
# 计算评分
scorer = ScheduleScorer(state, data)
report = scorer.score()

print(f"总分: {report.total_score:.1f} ({report.level.value})")
print()

# 各项指标
metrics_df = pd.DataFrame([
    {
        '指标': m.name,
        '得分': m.score,
        '权重': f"{m.weight:.0%}",
        '加权分': m.weighted_score,
        '等级': m.level.value
    }
    for m in report.metrics
])

display(metrics_df)

In [ ]:
# 可视化评分
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 雷达图数据
labels = [m.name for m in report.metrics]
scores = [m.score for m in report.metrics]

# 柱状图
ax1 = axes[0]
colors = ['#28a745' if s >= 80 else '#ffc107' if s >= 60 else '#dc3545' for s in scores]
bars = ax1.barh(labels, scores, color=colors)
ax1.set_xlim(0, 100)
ax1.set_xlabel('得分')
ax1.set_title('各项指标得分')
ax1.axvline(x=80, color='green', linestyle='--', alpha=0.5, label='良好线')
ax1.axvline(x=60, color='orange', linestyle='--', alpha=0.5, label='及格线')
ax1.legend()

# 添加数值标签
for bar, score in zip(bars, scores):
    ax1.text(score + 1, bar.get_y() + bar.get_height()/2, f'{score:.1f}', va='center')

# 饼图 - 加权分贡献
ax2 = axes[1]
weighted_scores = [m.weighted_score for m in report.metrics]
ax2.pie(weighted_scores, labels=labels, autopct='%1.1f%%')
ax2.set_title(f'加权分贡献 (总分: {report.total_score:.1f})')

plt.tight_layout()
plt.show()

## 5. 课表可视化

In [ ]:
# 生成热力图 - 每天各节次的排课数
daily_periods = defaultdict(lambda: defaultdict(int))

for record in state.schedule_records:
    daily_periods[record.day][record.period] += 1

# 转换为矩阵
matrix = []
for period in range(1, 10):
    row = [daily_periods[day][period] for day in range(1, 6)]
    matrix.append(row)

plt.figure(figsize=(10, 8))
sns.heatmap(
    matrix,
    xticklabels=['周一', '周二', '周三', '周四', '周五'],
    yticklabels=[f'第{i}节' for i in range(1, 10)],
    annot=True,
    fmt='d',
    cmap='YlOrRd'
)
plt.title('各时段排课数量热力图')
plt.xlabel('星期')
plt.ylabel('节次')
plt.show()

In [ ]:
# 班级课表示例
if data.classes:
    reporter = ScheduleReporter(state, data, report)
    first_class = data.classes[0]
    timetable = reporter.get_class_timetable(first_class.id)
    
    print(f"班级: {first_class.name}")
    print()
    
    # 转换为DataFrame
    days = ['周一', '周二', '周三', '周四', '周五']
    periods = [f'第{i}节' for i in range(1, 10)]
    
    table_data = []
    for period in range(1, 10):
        row = []
        for day in range(1, 6):
            cell = timetable[day][period]
            if cell:
                row.append(f"{cell.subject_name}\n{cell.teacher_name}")
            else:
                row.append('')
        table_data.append(row)
    
    df = pd.DataFrame(table_data, index=periods, columns=days)
    display(df.style.set_properties(**{'white-space': 'pre-wrap'}))

## 6. 参数调优实验

In [ ]:
# 运行多次实验，比较不同随机种子的结果
def run_experiment(seeds=[42, 123, 456, 789, 1000]):
    results = []
    
    for seed in seeds:
        # 生成数据
        config = MockConfig(
            grades=['G1', 'G2', 'G3', 'G4', 'G5', 'G6'],
            classes_per_grade=2,
            seed=seed
        )
        generator = MockDataGenerator()
        data = generator.generate(config)
        
        # 运行排课
        state, metrics = run_scheduling(data, verbose=False)
        
        # 评分
        scorer = ScheduleScorer(state, data)
        report = scorer.score()
        
        results.append({
            'seed': seed,
            'total_score': report.total_score,
            'completion_rate': next((m.score for m in report.metrics if m.name == '任务完成率'), 0),
            'morning_rate': next((m.score for m in report.metrics if m.name == '主科上午率'), 0),
            'time': metrics['total_time']
        })
        print(f"Seed {seed}: Score = {report.total_score:.1f}")
    
    return pd.DataFrame(results)

# 运行实验
print("运行多次实验...")
experiment_df = run_experiment()
print()
display(experiment_df)

In [ ]:
# 统计分析
print("实验统计:")
print(f"  平均分: {experiment_df['total_score'].mean():.2f}")
print(f"  标准差: {experiment_df['total_score'].std():.2f}")
print(f"  最高分: {experiment_df['total_score'].max():.2f}")
print(f"  最低分: {experiment_df['total_score'].min():.2f}")
print(f"  平均耗时: {experiment_df['time'].mean():.2f}s")

## 7. 问题诊断

In [ ]:
# 查看发现的问题
if report.issues:
    print("发现的问题:")
    for issue in report.issues:
        print(f"  - {issue}")
else:
    print("未发现明显问题")

print()

# 改进建议
if report.suggestions:
    print("改进建议:")
    for suggestion in report.suggestions:
        print(f"  - {suggestion}")

In [ ]:
# 教师工作量分析
teacher_workload = defaultdict(int)
for record in state.schedule_records:
    teacher_workload[record.teacher_name] += 1

workload_df = pd.DataFrame([
    {'教师': name, '周课时': hours}
    for name, hours in sorted(teacher_workload.items(), key=lambda x: x[1], reverse=True)
])

print("教师工作量分布:")
display(workload_df.head(10))

plt.figure(figsize=(12, 6))
plt.bar(range(len(workload_df)), workload_df['周课时'])
plt.axhline(y=workload_df['周课时'].mean(), color='r', linestyle='--', label='平均')
plt.xlabel('教师序号')
plt.ylabel('周课时')
plt.title('教师工作量分布')
plt.legend()
plt.show()

## 8. 导出结果

In [ ]:
# 生成完整报告
from app.engine.evaluation import ReportFormat

reporter = ScheduleReporter(state, data, report)

# Markdown 报告
md_report = reporter.generate_report(ReportFormat.MARKDOWN)
print(md_report)

In [ ]:
# 保存报告到文件
output_dir = os.path.join(project_root, 'notebooks', 'output')
os.makedirs(output_dir, exist_ok=True)

# 保存 Markdown
with open(os.path.join(output_dir, 'report.md'), 'w', encoding='utf-8') as f:
    f.write(md_report)

# 保存 JSON
json_report = reporter.generate_report(ReportFormat.JSON)
with open(os.path.join(output_dir, 'report.json'), 'w', encoding='utf-8') as f:
    f.write(json_report)

print(f"报告已保存到: {output_dir}")